# Recreating a Sankey Diagram Using ggplot2

**Analyst:** Itzel Castaneda Hernandez   
**Assignment:** Sankey Diagram Recreation  
**Date:** September, 2026

---

## 1. Introduction

The purpose of this assignment is to recreate a Sankey-style diagram originally provided in an Excel workbook using the R programming language and the `ggplot2` visualization framework.
The original visualization displays changes in five stroke risk factors across five time points: 1990, 1995, 2000, 2005, and 2010.
The objective of this analysis is to reproduce the appearance and structure of the original visualization as closely as possible. Particular attention is given to the ribbon colors, ribbon opacity, text color and placement, rounded shapes, and the amount of horizontal and vertical white space separating the graphical elements.
The visualization is constructed using polygon geometry rather than relying on a pre-built Sankey plotting function. This provides greater control over the appearance and layout of the individual ribbons and bands.

## 2. Packages

The project uses `ggplot2` for visualization and `dplyr` and `tidyr` for data manipulation.

In [19]:
library(ggplot2)
library(dplyr)
library(tidyr)

ERROR: there is no package called ‘dplyr’

## 3. Data

The dataset contains five risk factors measured across five years.

The risk factors are:

- Hypertension
- Diabetes
- Smoking
- Hypercholesterolemia
- Obesity

Each observation contains a risk factor, year index, and value. The year index is converted to the corresponding calendar year when the values are displayed in the final visualization.

In [ ]:
my_data <- read.csv("Stacked_column_stroke_blacks_noAF.csv")
head(my_data)

In [ ]:
years <- c(1990, 1995, 2000, 2005, 2010)

df <- rbind(
  data.frame(
    risk = "Hypertension",
    yr = 1:5,
    value = c(0.654, 0.633, 0.602, 0.561, 0.509)
  ),
  data.frame(
    risk = "Diabetes",
    yr = 1:5,
    value = c(0.359, 0.316, 0.260, 0.187, 0.092)
  ),
  data.frame(
    risk = "Smoking",
    yr = 1:5,
    value = c(0.171, 0.156, 0.142, 0.128, 0.116)
  ),
  data.frame(
    risk = "Hypercholesterolemia",
    yr = 1:5,
    value = c(0.161, 0.104, 0.045, 0.001, 0.001)
  ),
  data.frame(
    risk = "Obesity",
    yr = 1:5,
    value = c(0.001, 0.013, 0.043, 0.077, 0.115)
  )
)

df

## 4. Visualization Parameters

The original Excel visualization uses a consistent color for each risk factor. The colors below were manually specified to reproduce the appearance of the reference visualization.
Several additional parameters control the geometry of the plot. These include the minimum band height, spacing between bands, plateau width, connector spacing, corner rounding, and ribbon opacity.

In [ ]:
DRAW_ORDER <- c(
  "Hypertension",
  "Diabetes",
  "Smoking",
  "Obesity",
  "Hypercholesterolemia"
)

colors <- c(
  Hypertension = "#8BBB8F",
  Diabetes = "#DA654F",
  Smoking = "#7FABC5",
  Hypercholesterolemia = "#D99155",
  Obesity = "#4D6C96"
)

WHITE_SPACE <- 0.038
MIN_H <- 0.015

HALF_WIDTH <- 0.34
GAP <- 0.006

CORNER_X <- 0.015
CORNER_Y <- 0.010

RIBBON_OPACITY <- 1

FONT <- "sans"
SIZE_TITLE <- 18
SIZE_YEAR <- 14
SIZE_CAT <- 14
SIZE_VALUE <- 12

## 5. Creating the Stacked Bands

The first step in constructing the visualization is determining the vertical position of each risk-factor band at each year.
The original values are used to determine the relative height of each band. A small minimum height is added so that very small values remain visually represented in the plot.
Within each year, the bands are ordered by value. The cumulative heights are then used to calculate the lower boundary (`ymin`), upper boundary (`ymax`), and midpoint (`ymid`) of each band.
The midpoint is later used to position the numerical labels inside the bands.

In [ ]:
stack <- df %>%
  mutate(h = value + MIN_H) %>%
  group_by(yr) %>%
  arrange(value, risk, .by_group = TRUE) %>%
  mutate(
    rk = row_number(),
    ymin = cumsum(h) - h + GAP * (rk - 1),
    ymax = ymin + h,
    ymid = (ymin + ymax) / 2
  ) %>%
  ungroup()

stack

## 6. Creating Rounded Rectangles

Each year contains a set of horizontal plateau shapes representing the individual risk factors.
Because a standard `geom_rect()` does not provide the exact rounded-corner appearance required for the reference visualization, a custom function is used to generate the coordinates of each rounded rectangle.
The function creates four curved corners using points along circular arcs. These points are then combined to create the polygon representing each band.

In [ ]:
rrect <- function(xmin, xmax, ymin, ymax, rx, ry, n = 8) {

  ry <- min(ry, (ymax - ymin) / 2)

  arc <- function(cx, cy, a0, a1) {
    th <- seq(a0, a1, length.out = n)

    data.frame(
      x = cx + rx * cos(th),
      y = cy + ry * sin(th)
    )
  }

  rbind(
    arc(xmax - rx, ymax - ry, 0, pi / 2),
    arc(xmin + rx, ymax - ry, pi / 2, pi),
    arc(xmin + rx, ymin + ry, pi, 3 * pi / 2),
    arc(xmax - rx, ymin + ry, 3 * pi / 2, 2 * pi)
  )
}

In [ ]:
plateaus <- bind_rows(
  lapply(seq_len(nrow(stack)), function(i) {

    r <- stack[i, ]

    points <- rrect(
      r$yr - HALF_WIDTH,
      r$yr + HALF_WIDTH,
      r$ymin,
      r$ymax,
      CORNER_X,
      CORNER_Y
    )

    points$id <- paste0("p", i)
    points$risk <- r$risk

    points
  })
)

## 7. Creating the Connecting Ribbons

The next step is to connect each risk factor between consecutive years.
For each risk factor, the lower and upper boundaries of the band in one year are connected to the corresponding boundaries in the following year. The resulting four points form a polygon representing the sloped connector.
The connectors are drawn in a controlled order so that overlapping shapes reproduce the visual layering of the original diagram.

In [ ]:
slopes_tbl <- stack %>%
  group_by(risk) %>%
  arrange(yr, .by_group = TRUE) %>%
  mutate(
    ymin2 = lead(ymin),
    ymax2 = lead(ymax)
  ) %>%
  filter(!is.na(ymin2)) %>%
  ungroup() %>%
  arrange(match(risk, DRAW_ORDER))

In [ ]:
slopes <- bind_rows(
  lapply(seq_len(nrow(slopes_tbl)), function(i) {

    r <- slopes_tbl[i, ]

    xL <- r$yr + HALF_WIDTH + GAP
    xR <- r$yr + 1 - HALF_WIDTH - GAP

    data.frame(
      id = paste0("s", i),
      risk = r$risk,
      x = c(xL, xL, xR, xR),
      y = c(r$ymin, r$ymax, r$ymax2, r$ymin2)
    )
  })
)

slopes$id <- factor(
  slopes$id,
  levels = unique(slopes$id)
)

## 8. Preparing Labels

Two types of labels are used in the visualization.
The numerical values are displayed inside the colored bands and rounded to two decimal places. The risk-factor names are displayed on the left side of the visualization. The year labels are positioned above the plot.

In [ ]:
vlab <- stack %>%
  mutate(
    txt = sprintf(
      "%.2f",
      floor(value * 100 + 0.5 + 1e-9) / 100
    )
  )

catlab <- stack %>%
  filter(yr == 1)

## 9. Constructing the Final Plot

The final plot combines the connecting ribbons, yearly plateaus, numerical labels, risk-factor labels, and year labels.

The coordinate limits are manually specified so that the amount of white space surrounding the visualization is similar to the Excel reference chart.

In [ ]:
XLIM <- c(-0.277, 5.768)
YLIM <- c(-0.082, 2.017)

p <- ggplot() +

  geom_polygon(
    data = slopes,
    aes(
      x, y,
      group = id,
      fill = risk
    ),
    alpha = RIBBON_OPACITY
  ) +

  geom_polygon(
    data = plateaus,
    aes(
      x, y,
      group = id,
      fill = risk
    ),
    alpha = RIBBON_OPACITY
  ) +

  geom_text(
    data = vlab,
    aes(
      x = yr,
      y = ymid,
      label = txt
    ),
    colour = "white",
    fontface = "bold",
    size = SIZE_VALUE / .pt,
    family = FONT
  ) +

  geom_text(
    data = catlab,
    aes(
      x = XLIM[1] + 0.03,
      y = ymid,
      label = risk
    ),
    hjust = 0,
    colour = "black",
    size = SIZE_CAT / .pt,
    family = FONT
  ) +

  annotate(
    "text",
    x = 1:5,
    y = 1.634,
    label = years,
    colour = "black",
    size = SIZE_YEAR / .pt,
    family = FONT
  ) +

  annotate(
    "text",
    x = mean(XLIM),
    y = 1.936,
    label = "Risk Factors for Stroke in Blacks",
    fontface = "bold",
    size = SIZE_TITLE / .pt,
    family = FONT
  ) +

  scale_fill_manual(
    values = colors,
    guide = "none"
  ) +

  coord_cartesian(
    xlim = XLIM,
    ylim = YLIM,
    expand = FALSE,
    clip = "off"
  ) +

  theme_void() +

  theme(
    plot.margin = margin(0, 0, 0, 0),
    plot.background = element_rect(
      fill = "white",
      colour = "grey55",
      linewidth = 1
    )
  )

p

## 10. Final Sankey Diagram

The figure below represents the completed recreation of the Excel visualization using `ggplot2`.

The colored bands represent the five risk factors, while the connecting ribbons show how the relative values change between consecutive years. The numerical labels provide the underlying values for each risk factor and year.

<div align="center">
    <img src="sankey_diagram.png" width="50%">
</div>

## 11. Interpretation of the Result (Just for fun)

The resulting visualization shows several notable changes in the relative values of the risk factors over time.
Hypertension remains the largest band throughout the period, although its value decreases from 1990 to 2010. Diabetes also decreases substantially over the same period.
Smoking decreases more gradually, while hypercholesterolemia shows a large decline during the period. In contrast, obesity increases from a very small value in 1990 to a substantially larger value by 2010.
The Sankey-style structure makes these changes visible through both the changing widths of the bands and the movement of the corresponding ribbons between years.

## 12. Comparison With the Excel Visualization
Using custom polygon geometry provides greater control over the visualization than a standard bar chart or a basic Sankey implementation.
The R implementation was designed to reproduce the major visual characteristics of the original Excel visualization.
The following aspects were specifically controlled:

| Feature | Approach |
|---|---|
| Ribbon color | Manually specified hexadecimal colors |
| Ribbon opacity | Controlled with the `alpha` parameter |
| Numerical text | Centered within each band |
| Text color | White numerical labels and black category/year labels |
| Category labels | Positioned along the left side of the plot |
| Year labels | Positioned above the yearly plateaus |
| Rounded corners | Custom polygon geometry |
| Horizontal spacing | Controlled through plateau and connector coordinates |
| Vertical spacing | Controlled through band calculations |
| Plot boundaries | Manually specified coordinate limits |



## 13. Effect of Horizontal and Vertical White Space

The assignment specifically encouraged examining versions of the Sankey diagram with and without the horizontal and vertical white space separating the graphical elements.

Two parameters were used to control these spaces:

- `WHITE_SPACE` controls the vertical separation between adjacent bands within a given year.
- `GAP` controls the horizontal separation between a yearly plateau and its connecting ribbon.

Four versions of the visualization were considered. The first version contains both types of spacing, while the remaining versions progressively remove either the vertical spacing, horizontal spacing, or both.

| Version | `GAP` | `WHITE_SPACE` | Description |
|---|---:|---:|---|
| With both | 0.006 | 0.038 | Vertical and horizontal white space |
| No vertical | 0 | 0.038 | Bands touch within each year |
| No horizontal | 0.006 | 0 | Plateaus touch their connecting ribbons |
| No white space | 0 | 0 | Both types of spacing are removed |

These four configurations allow the visual effect of each spacing parameter to be examined independently.

### 13.1 With Both Types of White Space

The original configuration uses both vertical and horizontal spacing. This provides visual separation between adjacent bands and between the plateaus and their connecting ribbons.

<div align="center">
    <img src="sankey_diagram.png" width="50%">
</div>

### 13.2 Without Vertical White Space

In this version, `WHITE_SPACE` is set to zero. The bands within each year therefore touch one another vertically, while the horizontal separation between the plateaus and ribbons remains.

<div align="center">
    <img src="sankey_diagram_no_vertical.png" width="50%">
</div>

### 13.3 Without Horizontal White Space

In this version, `GAP` is set to zero. The connecting ribbons meet the yearly plateaus directly, while vertical spacing between the bands remains.

<div align="center">
    <img src="sankey_diagram_no_horizontal.png" width="50%">
</div>

### 13.4 Without White Space

Finally, both parameters are set to zero. The bands touch vertically and the connecting ribbons meet the plateaus directly.

<div align="center">
    <img src="sankey_diagram_no_space.png" width="50%">
</div>

## 14. Comparison with the Excel Chart

The recreated Sankey-style diagram was designed to match the original Excel chart as closely as possible using `ggplot2`. The main visual features reproduced include the ribbon colors, ribbon opacity, text colors, text placement, rounded shapes, and spacing between the graphical elements.

The additional whitespace versions demonstrate how the appearance of the diagram changes when the spacing between bands and connecting ribbons is removed.

<div align="center">
    <img src="Picture1.jpg" width="50%">
</div>

<div align="center">
    <img src="sankey_diagram.png" width=50%>
</div>

## 15. Conclusion

The Excel Sankey-style chart was successfully recreated using `ggplot2`. Custom polygon geometry was used to construct the rounded bands and connecting ribbons, while manually specified colors, text placement, opacity, and spacing were used to reproduce the appearance of the original visualization.

Four versions of the diagram were also created to demonstrate the effects of horizontal and vertical white space.

### Main Files Used

The main files used for this assignment are:

| File / Item                             | Description                                                                                          |
| --------------------------------------- | ---------------------------------------------------------------------------------------------------- |
| `Assignment No 2`                       | Assignment project and associated materials for recreating the Sankey-style diagram using `ggplot2`. |
| `notebooks_assignment_two_README.ipynb` | Jupyter Notebook/README for assignment two.                                                          |
| `Picture1.jpg`                          | Reference image used in the project.                                                                 |
| `sankey_diagram.R`                      | R script containing the code used to recreate the Sankey-style diagram.                              |
| `sankey_diagram.png`                    | Main recreated Sankey-style diagram.                                                                 |
| `sankey_diagram_no_horizontal.png`      | Sankey-style diagram without horizontal white space.                                                 |
| `sankey_diagram_no_vertical.png`        | Sankey-style diagram without vertical white space.                                                   |
| `sankey_diagram_no_space.png`           | Sankey-style diagram without horizontal or vertical white space.                                     |

